# Map creation
### Notebook creates a map for a specified vessel

### Library import

In [60]:
import os
import glob
import pandas as pd
import folium

### Parameter definition
#### Sets the date, vessel MMSI, and folder paths

In [69]:
DATE = "2026-02-27"
DATASET = f"aisdk-{DATE}"
MMSI = "219009229"
PROJECT_ROOT = os.path.abspath("..")
TASK2_OUTPUT = os.path.join(PROJECT_ROOT, "Task2", "output")
TASK3_OUTPUT = os.path.join(PROJECT_ROOT, "Task3", "output")

### File Search

In [70]:
folder = os.path.join(TASK2_OUTPUT, DATASET)
files = sorted(glob.glob(os.path.join(folder, "reduced_bucket_*.csv")))
files

['C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_0.csv',
 'C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_1.csv',
 'C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_2.csv',
 'C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_3.csv',
 'C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_4.csv',
 'C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_5.csv',
 'C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_6.csv',
 'C:\\Users\\Andrius\\PycharmProjects\\Big_Data_Task_1\\Task2\\output\\aisdk-2026-02-27\\reduced_bucket_7.csv']

### Loads the vessel positions from all bucket files

In [71]:
parts = []

for file in files:
    df = pd.read_csv(file)
    df["MMSI"] = df["MMSI"].astype(str)

    boat = df[df["MMSI"] == MMSI].copy()

    if len(boat) > 0:
        parts.append(boat[["MMSI", "timestamp", "latitude", "longitude"]])

track = pd.concat(parts, ignore_index=True)
track["timestamp"] = pd.to_datetime(track["timestamp"], errors="coerce")
track = track.dropna(subset=["timestamp", "latitude", "longitude"])
track = track.sort_values("timestamp").reset_index(drop=True)

track.head()

,MMSI,timestamp,latitude,longitude
0,219009229,2026-02-27 00:00:02,57.321493,11.126447
1,219009229,2026-02-27 00:00:12,57.321493,11.126447
2,219009229,2026-02-27 00:00:12,57.321493,11.126447
3,219009229,2026-02-27 00:00:21,57.321468,11.126462
4,219009229,2026-02-27 00:00:21,57.321468,11.126462


### Vessel/Boat path creation

In [72]:
center = [track["latitude"].mean(), track["longitude"].mean()]
m = folium.Map(location=center, zoom_start=6)

points = track[["latitude", "longitude"]].values.tolist()
folium.PolyLine(points, tooltip=f"MMSI {MMSI}").add_to(m)

### Hourly sampling
#### Creates an hourly sample to use for indentation within the map

In [73]:
track_hourly = (
    track
    .set_index("timestamp")
    .resample("1h")
    .first()
    .dropna(subset=["latitude", "longitude"])
    .reset_index()
)

track_hourly.head()

,timestamp,MMSI,latitude,longitude
0,2026-02-27 00:00:00,219009229,57.321493,11.126447
1,2026-02-27 01:00:00,219009229,57.321490,11.126597
2,2026-02-27 02:00:00,219009229,57.321505,11.126428
3,2026-02-27 03:00:00,219009229,57.321483,11.126557
4,2026-02-27 04:00:00,219009229,57.332662,11.211807


### Loads the vessel DFSI/Anomaly scores

In [74]:
scores_file = os.path.join(TASK3_OUTPUT, DATASET, "final_scores.csv")
scores = pd.read_csv(scores_file)
scores["MMSI"] = scores["MMSI"].astype(str)

score = scores[scores["MMSI"] == MMSI].iloc[0]
score

MMSI                         219009229
A_count                              0
B_count                             35
C_count                              0
D_count                             28
max_gap_hours                      0.0
total_impossible_jump_nm    17971.8797
DFSI                          1797.188
Name: 539, dtype: object

### Final map creation

In [75]:
center = [track["latitude"].mean(), track["longitude"].mean()]
m = folium.Map(location=center, zoom_start=6)

points = track[["latitude", "longitude"]].values.tolist()
folium.PolyLine(points, tooltip=f"MMSI {MMSI}").add_to(m)

for _, row in track_hourly.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        popup=str(row["timestamp"]),
        tooltip=str(row["timestamp"]),
        fill=True,
    ).add_to(m)

start = track.iloc[0]
end = track.iloc[-1]

folium.Marker(
    [start["latitude"], start["longitude"]],
    popup=f"Start<br>{start['timestamp']}",
    tooltip="Start",
).add_to(m)

folium.Marker(
    [end["latitude"], end["longitude"]],
    popup=f"End<br>{end['timestamp']}",
    tooltip="End",
).add_to(m)

legend_html = f"""
<div style="
    position: fixed;
    bottom: 20px;
    left: 20px;
    z-index: 9999;
    background-color: white;
    border: 2px solid grey;
    border-radius: 8px;
    padding: 12px;
    font-size: 14px;
">
<b>Boat info</b><br>
MMSI: {MMSI}<br>
DFSI: {score['DFSI']}<br>
A anomaly count: {int(score['A_count'])}<br>
B anomaly count: {int(score['B_count'])}<br>
C anomaly count: {int(score['C_count'])}<br>
D anomaly count: {int(score['D_count'])}
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

m.fit_bounds(points)
m

In [76]:
map_name = f"{DATE}_{MMSI}_map.html"
m.save(map_name)
print("Saved:", map_name)

Saved: 2026-02-27_219009229_map.html
